# 🚀 Day 04a: Optimizers & Regularization — Train Better, Faster

---

## 🎯 What You'll Master Today
- SGD, Momentum, RMSProp, Adam — from scratch
- L1 / L2 regularization
- Dropout & Batch Normalization
- Learning rate scheduling

**Goal:** Implement Adam from memory. Know when to use each regularization technique.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  OPTIMIZERS — Evolution                                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  SGD:      W -= lr * g           (vanilla, oscillates)          ║
║  Momentum: v = β·v + g           (smooths oscillations)         ║
║            W -= lr * v                                           ║
║  RMSProp:  s = β·s + (1-β)·g²    (per-param learning rate)     ║
║            W -= lr * g / √(s+ε)                                 ║
║  Adam:     momentum + RMSProp + bias correction                 ║
║            (the default choice)                                   ║
║                                                                   ║
║  REGULARIZATION:                                                 ║
║    L2 (weight decay)  → penalize large weights                  ║
║    L1 (sparsity)      → push weights to zero                    ║
║    Dropout            → randomly zero neurons (training only)   ║
║    Batch Norm         → normalize activations per batch         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# 1. SGD — The Foundation
# ============================================================

class SGD:
    def __init__(self, lr=0.01):
        self.lr = lr
    
    def update(self, params, grads):
        for p, g in zip(params, grads):
            p -= self.lr * g

# Test on a simple quadratic: f(x) = x²
x = np.array([5.0])
sgd = SGD(lr=0.1)

print("SGD on f(x) = x², starting at x=5:")
for i in range(10):
    grad = 2 * x  # df/dx = 2x
    sgd.update([x], [grad])
    if i % 2 == 0:
        print(f"  Step {i}: x = {x[0]:.6f}, f(x) = {x[0]**2:.6f}")

In [ ]:
# ============================================================
# 2. SGD with Momentum — Smooth out oscillations
# ============================================================

class SGDMomentum:
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.velocities = None
    
    def update(self, params, grads):
        if self.velocities is None:
            self.velocities = [np.zeros_like(p) for p in params]
        
        for i, (p, g) in enumerate(zip(params, grads)):
            self.velocities[i] = self.beta * self.velocities[i] + g
            p -= self.lr * self.velocities[i]

# Compare SGD vs Momentum on a 2D problem
# f(x,y) = x² + 100y² (elongated valley — SGD oscillates badly)
def rosenbrock_grad(pos):
    return np.array([2*pos[0], 200*pos[1]])

# SGD path
pos_sgd = np.array([5.0, 5.0])
sgd = SGD(lr=0.001)
for _ in range(100):
    sgd.update([pos_sgd], [rosenbrock_grad(pos_sgd)])

# Momentum path
pos_mom = np.array([5.0, 5.0])
mom = SGDMomentum(lr=0.001, beta=0.9)
for _ in range(100):
    mom.update([pos_mom], [rosenbrock_grad(pos_mom)])

print(f"After 100 steps on elongated valley:")
print(f"  SGD:      ({pos_sgd[0]:.4f}, {pos_sgd[1]:.4f})")
print(f"  Momentum: ({pos_mom[0]:.4f}, {pos_mom[1]:.4f})")
print(f"\n💡 Momentum accelerates in consistent direction, damps oscillations.")

In [ ]:
# ============================================================
# 3. Adam — The Default Choice (MEMORIZE THIS)
# ============================================================

class Adam:
    """
    Adam = Momentum (m) + RMSProp (v) + bias correction.
    
    m = β1·m + (1-β1)·g          ← 1st moment (mean of gradients)
    v = β2·v + (1-β2)·g²         ← 2nd moment (variance of gradients)
    m_hat = m / (1-β1^t)          ← bias correction
    v_hat = v / (1-β2^t)          ← bias correction
    W -= lr * m_hat / (√v_hat + ε)
    """
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = None  # 1st moment
        self.v = None  # 2nd moment
        self.t = 0     # timestep
    
    def update(self, params, grads):
        if self.m is None:
            self.m = [np.zeros_like(p) for p in params]
            self.v = [np.zeros_like(p) for p in params]
        
        self.t += 1
        
        for i, (p, g) in enumerate(zip(params, grads)):
            # Update moments
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2
            
            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            
            # Update
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Test Adam
pos_adam = np.array([5.0, 5.0])
adam = Adam(lr=0.1)

for i in range(100):
    adam.update([pos_adam], [rosenbrock_grad(pos_adam)])
    if i % 20 == 0:
        print(f"  Step {i}: ({pos_adam[0]:.6f}, {pos_adam[1]:.6f})")

print(f"\nAdam converges to: ({pos_adam[0]:.6f}, {pos_adam[1]:.6f})")
print("\n💡 Adam = adaptive learning rate per parameter. Almost always the best starting choice.")

In [ ]:
# ============================================================
# 4. L1 & L2 Regularization — From Scratch
# ============================================================

# L2 Regularization (weight decay)
# Loss_total = Loss_data + (λ/2m) * Σ(w²)
# Gradient: dW += (λ/m) * W

def l2_regularization(W, lambd, m):
    """Returns regularization loss and gradient."""
    reg_loss = (lambd / (2*m)) * np.sum(W**2)
    reg_grad = (lambd / m) * W
    return reg_loss, reg_grad

# L1 Regularization (sparsity)
# Loss_total = Loss_data + (λ/m) * Σ|w|
# Gradient: dW += (λ/m) * sign(W)

def l1_regularization(W, lambd, m):
    reg_loss = (lambd / m) * np.sum(np.abs(W))
    reg_grad = (lambd / m) * np.sign(W)
    return reg_loss, reg_grad

# Compare
W = np.array([0.5, -0.1, 3.0, 0.01, -2.0])
lambd = 0.1
m = 100

l2_loss, l2_grad = l2_regularization(W, lambd, m)
l1_loss, l1_grad = l1_regularization(W, lambd, m)

print(f"Weights: {W}")
print(f"\nL2 reg grad: {l2_grad.round(6)}  ← proportional to weight size")
print(f"L1 reg grad: {l1_grad.round(6)}  ← constant magnitude, pushes to 0")
print(f"\n💡 L2 shrinks all weights. L1 kills small weights → sparsity.")

In [ ]:
# ============================================================
# 5. Dropout — From Scratch
# ============================================================

def dropout_forward(A, drop_rate=0.5, training=True):
    """
    Inverted dropout: scale during training so test is unchanged.
    """
    if not training:
        return A
    
    # Create mask: 1 with prob (1-drop_rate), 0 with prob drop_rate
    mask = (np.random.rand(*A.shape) > drop_rate).astype(float)
    
    # Scale by 1/(1-p) to keep expected value same
    A_dropped = A * mask / (1 - drop_rate)
    
    return A_dropped, mask

# Demo
np.random.seed(42)
A = np.ones((1, 10))  # all 1s for clarity

A_train, mask = dropout_forward(A, drop_rate=0.3, training=True)
A_test = dropout_forward(A, drop_rate=0.3, training=False)

print(f"Original:  {A[0]}")
print(f"Mask:      {mask[0].astype(int)}")
print(f"Training:  {A_train[0].round(4)}  (scaled by 1/0.7 ≈ 1.43)")
print(f"Testing:   {A_test[0]}  (untouched)")
print(f"\nExpected value preserved:")
print(f"  Training mean: {A_train.mean():.3f}")
print(f"  Testing mean:  {A_test.mean():.3f}")

print("\n💡 Inverted dropout: scale at train time → no change at test time.")
print("   Typical: p=0.5 for FC layers, p=0.2 for input, p=0 for output.")

In [ ]:
# ============================================================
# 6. Batch Normalization — From Scratch
# ============================================================

class BatchNorm:
    """
    Normalize activations: x_hat = (x - μ_batch) / √(σ²_batch + ε)
    Then scale and shift: y = γ * x_hat + β  (learnable)
    """
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.gamma = np.ones(num_features)    # learnable scale
        self.beta = np.zeros(num_features)     # learnable shift
        self.eps = eps
        self.momentum = momentum
        # Running stats for inference
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)
    
    def forward(self, X, training=True):
        if training:
            # Use batch statistics
            mean = X.mean(axis=0)
            var = X.var(axis=0)
            
            # Update running stats
            self.running_mean = (1-self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1-self.momentum) * self.running_var + self.momentum * var
        else:
            # Use running statistics (accumulated during training)
            mean = self.running_mean
            var = self.running_var
        
        # Normalize
        x_hat = (X - mean) / np.sqrt(var + self.eps)
        
        # Scale and shift
        out = self.gamma * x_hat + self.beta
        return out

# Demo
np.random.seed(42)
X = np.random.randn(32, 4) * 10 + 50  # mean≈50, std≈10

bn = BatchNorm(num_features=4)
X_norm = bn.forward(X, training=True)

print(f"Before BN:  mean={X.mean(axis=0).round(2)}, std={X.std(axis=0).round(2)}")
print(f"After BN:   mean={X_norm.mean(axis=0).round(4)}, std={X_norm.std(axis=0).round(4)}")

print("\n💡 BatchNorm benefits:")
print("  • Stabilizes training (reduces internal covariate shift)")
print("  • Allows higher learning rates")
print("  • Acts as mild regularization")
print("  • Training: batch stats. Inference: running stats!")

---

## 🗺️ Optimizers & Regularization Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  OPTIMIZER SELECTION                                             ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Default:  Adam (lr=0.001, β1=0.9, β2=0.999)                   ║
║  Vision:   SGD + Momentum + LR scheduler (often better final)   ║
║  NLP:      AdamW (Adam + decoupled weight decay)                ║
║                                                                   ║
║  REGULARIZATION                                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  L2 / Weight Decay  → default, shrinks all weights              ║
║  L1 / Lasso         → sparsity, feature selection               ║
║  Dropout            → p=0.5 for FC, p=0.1-0.3 for others       ║
║  Batch Norm         → after linear, before activation           ║
║  Early Stopping     → stop when val loss stops improving        ║
║  Data Augmentation  → increase effective dataset size           ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Explain Adam optimizer | Combines momentum (1st moment) and RMSProp (2nd moment) with bias correction. Adaptive per-param lr |
| 2 | L1 vs L2 regularization? | L1: pushes weights to exactly 0 (sparsity). L2: shrinks all weights proportionally (no zeros) |
| 3 | How does dropout work? | Randomly zero activations during training with prob p. Scale by 1/(1-p). No dropout at test time |
| 4 | What does batch norm do? | Normalizes activations per batch (mean=0, var=1), then learns scale/shift. Uses running stats at inference |
| 5 | Adam vs SGD? | Adam: fast convergence, less tuning. SGD+momentum: often better generalization (with proper LR schedule) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Adam = default optimizer (momentum + RMSProp)        │
## │  • L2 shrinks weights, L1 creates sparsity              │
## │  • Dropout: inverted scaling → train/test consistency   │
## │  • BatchNorm: batch stats (train), running stats (test) │
## │  • Combine: Adam + dropout + BN + early stopping        │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **PyTorch Tensors & Autograd** — where all this math becomes code